In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
gold = pd.read_parquet(repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet")

## Mean chl-a over time by polarity

Eddy-interior T chl-a for every eddy-day (faint points) with the bi-weekly mean +/- 1 SD (line), cyclones and anticyclones side by side.
The seasonal cycle (spring bloom) shows in both, and the two polarities track each other closely - consistent with the near-equal overall means.

In [ ]:
import matplotlib.dates as mdates

gold = pd.read_parquet(repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet")
gold["date"] = pd.to_datetime(gold["date"])

panels = [(1, "Cyclone", "#4c72b0"), (0, "Anticyclone", "#c44e52")]
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, (polarity, title, color) in zip(axes, panels):
    sub = gold[gold["polarity"] == polarity]
    ax.scatter(sub["date"], sub["eddy_mean_Tchla"], s=14, color=color, alpha=0.2, edgecolors="none")
    binned = sub.set_index("date")["eddy_mean_Tchla"].groupby(pd.Grouper(freq="2W"))
    mean, std, n = binned.mean(), binned.std(), binned.count()
    keep = n >= 5  # drop sparse end bins that would spike the line
    ax.errorbar(mean[keep].index, mean[keep].values, yerr=std[keep].values, color=color,
                lw=2, marker="o", ms=6, capsize=3, elinewidth=1.2)
    ax.set_title(title, color=color, fontsize=15, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, ls="--")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
axes[0].set_ylabel("Mean T chl-a (mg m$^{-3}$)")
axes[0].set_ylim(bottom=0)
fig.suptitle("Mean Chl-a Over Time by Polarity", fontsize=17, fontweight="bold")
fig.tight_layout()
plt.show()

## Where the high-chl outliers sit

The faint high-chl points in the time series above are not spread evenly across the region.
Each eddy-day center is mapped and colored by interior T chl-a, with the outliers (> 0.8 mg m$^{-3}$) ringed. For both polarities the outliers cluster on the inshore, shelf-slope side of the Gulf Stream. They sit off Cape Hatteras, along the New England shelf-break, and on the northern slope.
That is the productive side of the front, so a high interior value mostly reflects where the eddy sat rather than its polarity.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

outlier_threshold = 0.8  # mg m^-3; the high tail of the time series above
gold["is_outlier"] = gold["eddy_mean_Tchla"] > outlier_threshold

# Median Gulf Stream path over the study window, to read inshore vs offshore.
sl = pd.read_parquet(repo_root / "data" / EXPERIMENT / "silver" / "gulf_stream" / "streamline.parquet")
gs_path = sl[sl["date"] >= "2024-10-01"].groupby("point_idx")[["lon", "lat"]].median()

fig = plt.figure(figsize=(15, 6.5))
for i, (polarity, title) in enumerate([(1, "Cyclone"), (0, "Anticyclone")]):
    ax = fig.add_subplot(1, 2, i + 1, projection=ccrs.PlateCarree())
    ax.set_extent([-80, -54, 28, 44], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="#e8e6e1", zorder=0)
    ax.add_feature(cfeature.COASTLINE, lw=0.6, edgecolor="0.4")
    ax.plot(gs_path["lon"], gs_path["lat"], color="0.25", lw=1.6, ls="--",
            transform=ccrs.PlateCarree(), label="Gulf Stream (median path)")
    sub = gold[gold["polarity"] == polarity]
    # Join each eddy's daily centers in time order so one track reads as one eddy.
    for j, (_, track) in enumerate(sub.sort_values("date").groupby("track_id")):
        ax.plot(track["center_lon"], track["center_lat"], color="0.55", lw=0.6, alpha=0.4,
                zorder=0.6, transform=ccrs.PlateCarree(),
                label="eddy track (time-ordered)" if i == 0 and j == 0 else None)
    base, high = sub[~sub["is_outlier"]], sub[sub["is_outlier"]]
    sc = ax.scatter(base["center_lon"], base["center_lat"], c=base["eddy_mean_Tchla"],
                    cmap="viridis", vmin=0, vmax=1.0, s=20, alpha=0.55, edgecolors="none",
                    transform=ccrs.PlateCarree())
    ax.scatter(high["center_lon"], high["center_lat"], c=high["eddy_mean_Tchla"],
               cmap="viridis", vmin=0, vmax=1.0, s=85, edgecolors="black", lw=0.9,
               zorder=5, transform=ccrs.PlateCarree())
    ax.set_title(f"{title}  ({len(high)} eddy-days > {outlier_threshold} mg m$^{{-3}}$)", fontsize=13)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color="0.85")
    gl.top_labels = gl.right_labels = False
    if i == 0:
        ax.legend(loc="lower left", fontsize=9, framealpha=0.9)
cbar = fig.colorbar(sc, ax=fig.axes, shrink=0.7, pad=0.02)
cbar.set_label("Eddy interior mean T chl-a (mg m$^{-3}$)")
fig.suptitle("Location of high-chl eddy-days (outliers ringed, joined by eddy track)", fontsize=15, fontweight="bold")
plt.show()

## Pigment concentration distributions by polarity

Per-eddy-day interior mean for each of the 13 SDP pigments, cyclone (blue) vs anticyclone (red), density-normalized.
The two distributions sit almost on top of each other for every pigment - the same near-equal-by-polarity result as the means and the time series above.

In [ ]:
from matplotlib.patches import Patch

pigments = [c.removeprefix("eddy_mean_") for c in gold.columns if c.startswith("eddy_mean_")]
ncols = 4
nrows = -(-len(pigments) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 2.8 * nrows))
for ax, pigment in zip(axes.flat, pigments):
    for pol, color in [(1, "#2166ac"), (0, "#b2182b")]:
        ax.hist(gold.loc[gold["polarity"] == pol, f"eddy_mean_{pigment}"], bins=30,
                alpha=0.5, color=color, density=True)
    ax.set_title(pigment.replace("_", " "))
    ax.set_xlabel("mg / m\u00b3")
    ax.set_ylabel("Probability Density")
    ax.spines[["top", "right"]].set_visible(False)

# Legend goes in the leftover grid cells (13 pigments leave 3 empty), then hide them.
legend_handles = [Patch(facecolor="#2166ac", alpha=0.5, label="Cyclone"),
                  Patch(facecolor="#b2182b", alpha=0.5, label="Anticyclone")]
leftover = axes.flat[len(pigments):]
leftover[0].legend(handles=legend_handles, loc="center", frameon=False, fontsize=14)
for ax in leftover:
    ax.axis("off")
fig.suptitle("Pigment Concentration Distributions: Cyclone vs Anticyclone", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Radial pigment structure by polarity

For each pigment, the symmetric relative difference between cyclones and anticyclones, 200*(cyc - anti)/(cyc + anti), versus normalized distance from the eddy center (r / R), with 95% bootstrap CIs.
Blue means cyclones are higher at every radius, red anticyclones, grey not significant.

8-day compositing degrades this center-vs-edge signal most, because the eddy drifts across the window. Radial analysis was deferred for this experiment, so read the result as directional only.
The point estimates do lean the expected way. Cyclones are ~6-12% higher at the center, fading to ~0 at the edge, for almost every pigment. DV chla, the Prochlorococcus marker, is the flat exception. Every band crosses zero, so none of the differences is significant here.
That is also why the bulk means look equal by polarity: the cyclone signal sits at the center, and bulk-averaging dilutes it.

In [ ]:
import glob

RADIAL_BINS = [0, 0.25, 0.5, 0.75, 1.0, 1.5]
bin_mids = np.array([(RADIAL_BINS[i] + RADIAL_BINS[i + 1]) / 2 for i in range(len(RADIAL_BINS) - 1)])
bin_labels = [f"{RADIAL_BINS[i]:.2f}-{RADIAL_BINS[i + 1]:.2f}" for i in range(len(RADIAL_BINS) - 1)]

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

# Per-pixel pigments need each eddy-day's radius (from the gold table) to normalize r.
pigment_dir = repo_root / "data" / EXPERIMENT / "silver" / "pigments"
frames = []
for pol, val in [("cyclone", 1), ("anticyclone", 0)]:
    for fp in glob.glob(str(pigment_dir / pol / "*.parquet")):
        df = pd.read_parquet(fp)
        df["polarity"], df["pol_int"] = pol, val
        frames.append(df)
pix = pd.concat(frames, ignore_index=True)
pix["date"] = pd.to_datetime(pix["date"])

radius = gold[["track_id", "date", "polarity", "radius_km"]].rename(columns={"polarity": "pol_int"})
radius["date"] = pd.to_datetime(radius["date"])
pix = pix.merge(radius, on=["track_id", "date", "pol_int"], how="left")
pix["r_norm"] = haversine_km(pix["pixel_lon"].values, pix["pixel_lat"].values,
                             pix["center_lon"].values, pix["center_lat"].values) / pix["radius_km"]
pix["r_bin"] = pd.cut(pix["r_norm"], bins=RADIAL_BINS, labels=bin_labels, right=False)
binned = pix.dropna(subset=["r_bin", "radius_km"])

def radial_difference(df, col, n_boot=1000, seed=42):
    """Per radial bin: 200*(cyc - anti)/(cyc + anti) with 95% bootstrap CI across eddy-days."""
    grouped = df.groupby(["track_id", "date", "r_bin", "polarity"], observed=True)[col].mean().reset_index()
    rng = np.random.default_rng(seed)
    diffs, lo, hi = [], [], []
    for label in bin_labels:
        cyc = grouped.loc[(grouped.r_bin == label) & (grouped.polarity == "cyclone"), col].values
        anti = grouped.loc[(grouped.r_bin == label) & (grouped.polarity == "anticyclone"), col].values
        if len(cyc) < 3 or len(anti) < 3:
            diffs.append(np.nan); lo.append(np.nan); hi.append(np.nan); continue
        boots = np.empty(n_boot)
        for b in range(n_boot):
            c = rng.choice(cyc, len(cyc), replace=True).mean()
            a = rng.choice(anti, len(anti), replace=True).mean()
            boots[b] = 200 * (c - a) / (c + a)
        diffs.append(200 * (cyc.mean() - anti.mean()) / (cyc.mean() + anti.mean()))
        lo.append(np.percentile(boots, 2.5)); hi.append(np.percentile(boots, 97.5))
    return np.array(diffs), np.array(lo), np.array(hi)

pigment_cols = [c for c in ["T chla", "Zea", "DV chla", "ButFuco", "HexFuco", "Allo", "MV chlb",
                            "Neo", "Viola", "Fuco", "chl c1+c2", "chl c3", "Perid"] if c in pix.columns]
ncols = 4
nrows = -(-len(pigment_cols) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 2.9 * nrows))
for i, (ax, pig) in enumerate(zip(axes.flat, pigment_cols)):
    diffs, lo, hi = radial_difference(binned, pig)
    valid = ~np.isnan(lo)
    color = "#2166ac" if valid.any() and np.all(lo[valid] > 0) else "#b2182b" if valid.any() and np.all(hi[valid] < 0) else "0.4"
    ax.plot(bin_mids, diffs, "o-", color=color, ms=5, lw=1.5)
    ax.fill_between(bin_mids, lo, hi, color=color, alpha=0.2)
    ax.axhline(0, color="0.5", ls="--", lw=0.8)
    ax.set_title(pig)
    ax.set_xlabel("Normalized Radius (r / R)")
    if i % ncols == 0:
        ax.set_ylabel("Symmetric Relative Diff (%)")
    ax.set_xlim(0, 1.5)
    ax.spines[["top", "right"]].set_visible(False)
for ax in axes.flat[len(pigment_cols):]:
    ax.set_visible(False)
fig.suptitle("Radial Pigment Structure: Cyclone vs Anticyclone (8-day composite data)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()